# Phase 5: Portfolio Optimization

Objective:
To construct an optimal portfolio allocation using
mean-variance optimization.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [2]:
tickers = ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS"]

data = yf.download(tickers, period="5y")["Close"]

data.head()

[*********************100%***********************]  4 of 4 completed


Ticker,HDFCBANK.NS,INFY.NS,RELIANCE.NS,TCS.NS
Date,,,,
2021-03-02,742.680359,1148.377808,954.926147,2642.813477
2021-03-03,751.512756,1182.754395,998.500916,2689.184570
2021-03-04,735.031982,1171.134155,986.598450,2680.920898
2021-03-05,724.589355,1159.117798,987.890625,2644.307373
2021-03-08,719.616760,1175.887817,993.513306,2643.340576


In [3]:
returns = data.pct_change().dropna()

returns.head()

Ticker,HDFCBANK.NS,INFY.NS,RELIANCE.NS,TCS.NS
Date,,,,
2021-03-03,0.011893,0.029935,0.045632,0.017546
2021-03-04,-0.021930,-0.009825,-0.011920,-0.003073
2021-03-05,-0.014207,-0.010260,0.001310,-0.013657
2021-03-08,-0.006863,0.014468,0.005692,-0.000366
2021-03-09,0.028299,0.007337,-0.000023,0.014633


In [4]:
mean_returns = returns.mean() * 252
cov_matrix = returns.cov() * 252

print("Annual Returns:\n", mean_returns)

Annual Returns:
 Ticker
HDFCBANK.NS    0.055520
INFY.NS        0.053169
RELIANCE.NS    0.096837
TCS.NS         0.019713
dtype: float64


In [5]:
def portfolio_performance(weights, mean_returns, cov_matrix):
    returns = np.dot(weights, mean_returns)
    volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return returns, volatility

In [6]:
def negative_sharpe(weights, mean_returns, cov_matrix):
    p_return, p_vol = portfolio_performance(weights, mean_returns, cov_matrix)
    return -p_return / p_vol

num_assets = len(mean_returns)
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0, 1) for asset in range(num_assets))
initial_weights = num_assets * [1./num_assets]

optimized = minimize(
    negative_sharpe,
    initial_weights,
    args=(mean_returns, cov_matrix),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

optimal_weights = optimized.x
optimal_weights

array([2.12100203e-01, 1.47169803e-01, 6.40729995e-01, 8.64312461e-18])

In [7]:
portfolio = pd.DataFrame({
    "Stock": tickers,
    "Optimal Weight": optimal_weights
})

portfolio

,Stock,Optimal Weight
0,RELIANCE.NS,2.121002e-01
1,TCS.NS,1.471698e-01
2,HDFCBANK.NS,6.407300e-01
3,INFY.NS,8.643125e-18
